In [ ]:
import torch
import torch.optim as optim
from torch.optim.lr_scheduler import OneCycleLR
from torch.utils.data import DataLoader
from scripts.custom_dataset import CustomDataset
from model.model import MHAModel
import pandas as pd
import json
import time
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

from sklearn.metrics import precision_recall_fscore_support

In [2]:
if sys.platform == 'linux':
    load_dotenv(dotenv_path=(Path('.')/'.env.linux'))
elif sys.platform == 'win32':
    load_dotenv(dotenv_path=(Path('.')/'.env.win'))
else:
    raise ValueError('Ваша операционная система не поддерживается!')

DATASET_VERSION = os.getenv('DATASET_VERSION', '2.16') # Допустимые занчения: 2.3; 2.16 | В версии 2.3 меньше тренировочных примеров, по сравнению с 2.16. Точность на тестовой выборке практически не меняется
DATASET_PATH = os.getenv('DATASET_PATH')
CORPUS_TEXTS_FILEPATH = os.getenv('CORPUS_TEXTS_FILEPATH')

In [3]:
USE_CLASSES_WEIGHTS = False
CLASSES_WEIGHTS_SCALER = 12

SHUFFLE = True
DROP_LAST = True
EPOCHS = 5
LEARNING_RATE = 5e-5
# MAX_LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5

USE_PRETRAINED = True

BATCH_SIZE = 128
INIT_WEIGHTS = True
BIAS = True
TOKENS_EMBEDDING_DIM = 256
LETTERS_EMBEDDING_DIM = 32 # Важно, если WORD_REPRESENTATION = 'both', то ATTENTION_DIM = (LETTERS_EMBEDDING_DIM * MAX_LETTERS_COUNT) + TOKENS_EMBEDDING_DIM
LETTERS_IN_WORD_ATTENTION_DIM = 128
MAIN_ATTENTION_DIM = 512
MAIN_NUM_HEADS = 8
MAIN_NUM_ENCODER_LAYERS = 4
MAIN_ENCODER_FC_HIDDEN_DIM = MAIN_ATTENTION_DIM*4 # Как в классическом трансформере

CLASSIFIER_FC_HIDDEN_DIM = MAIN_ATTENTION_DIM*2

WORD_REPRESENTATION = 'tokens' # tokens; letters; both  Уровень представления слова (токены, буквы, токены + буквы)
WORDS_POS_ENCODING = 'learnable' # Допустимые значения: sin; learnable; None
WORD_SUBTOKENS_POS_ENCODING = 'rope' # Допустимые значения: learnable; None
LETTERS_POS_ENCODING = 'rope' # Допустимые значения: learnable; sin; None
ROPE_BASE = 10000

DROPOUT = 0.25
TEMPERATURE = 1
BATCH_FIRST = True

MODEL_SAVE_FILEPATH = f'model/{WORD_REPRESENTATION}_model_params.pt'

RANDOM_STATE = 42

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
# DEVICE = 'cpu'

In [4]:
print(torch.backends.cuda.flash_sdp_enabled())
print(torch.backends.cuda.mem_efficient_sdp_enabled())
print(torch.backends.cuda.math_sdp_enabled())

True
True
True


In [5]:
def generate_batches(dataset:CustomDataset, batch_size:int, shuffle:bool=True, drop_last:bool=True, device='cpu'):
    '''Создает батчи из датасета и переносит данные на девайс'''
    dataloader = DataLoader(dataset, batch_size, shuffle, drop_last=drop_last)
    for data_dict in dataloader:
        out_data_dict = {}
        for name, _ in data_dict.items():
            out_data_dict[name] = data_dict[name].to(device)
        yield out_data_dict

In [6]:
def save_results_to_file(model, model_filepath:str, train_states:list=None, validation_states:list=None):
    '''Сохраняет параметры модели и метрики обучения в файлы'''
    torch.save(model, model_filepath)
    if train_states is not None:
        with open(f"data/{WORD_REPRESENTATION}_train_states.json", "w", encoding="utf-8") as file:
            json.dump(train_states, file, indent=4, ensure_ascii=False)
        
        with open(f"model/{WORD_REPRESENTATION}_model_hyperparams.json", "w", encoding="utf-8") as file:
            json.dump(model.get_hyperparams(), file, indent=4, ensure_ascii=False)

    if validation_states is not None:
        with open(f"data/{WORD_REPRESENTATION}_validation_states.json", "w", encoding="utf-8") as file:
            json.dump(validation_states, file, indent=4, ensure_ascii=False)

In [7]:
def normalize_sizes(predictions:dict[str:torch.tensor], targets:dict[str:torch.tensor], target_names:list[str]):
    for key in target_names:
        # Для predictions: [B, S, C] -> [B*S, C]
        if len(predictions[key].size()) == 3:
            predictions[key] = predictions[key].contiguous().view(-1, predictions[key].size(-1))
        
        # Для targets: [B, S] -> [B*S]
        if len(targets[key].size()) == 2:
            targets[key] = targets[key].contiguous().view(-1)
    
    return predictions, targets

In [8]:
def reveal_mistakes(predictions:dict[str:torch.tensor], targets:dict[str:torch.tensor], target_names:list[str], pad_idx:int=0) -> dict[str:torch.Tensor]:
    incorrect_indices = {}
    for key in target_names:
        _, pred_indices = predictions[key].max(dim=-1)
        correct_indices = torch.eq(pred_indices, targets[key])
        valid_indices = torch.ne(targets[key], pad_idx)
        incorrect_indices[key] = (~(correct_indices*valid_indices))*valid_indices  # bool tensor
    return incorrect_indices

In [9]:
# def compute_loss(predictions:dict[str:torch.tensor], targets:dict[str:list[int]], target_names:list[str], target_weights:dict[str:float], pad_idx:int=0):
def compute_loss(predictions:dict[str:torch.tensor], targets:dict[str:list[int]], target_names:list[str], pad_idx:int=0):
    predictions, targets = normalize_sizes(predictions, targets, target_names)
    losses = {}
    total_loss = 0
    for key in target_names:
        # target_weights[key]['classes_weights'] = target_weights[key]['classes_weights'].to(DEVICE)
        # losses[key] = torch.nn.functional.cross_entropy(predictions[key], targets[key], weight=target_weights[key]['classes_weights'], ignore_index=pad_idx)
        losses[key] = torch.nn.functional.cross_entropy(predictions[key], targets[key], ignore_index=pad_idx)
        # total_loss += losses[key] * target_weights[key]['grammem_weight']
        total_loss += losses[key]

    return total_loss, losses

In [10]:
def compute_metrics(predictions: dict[str, torch.Tensor], targets: dict[str, torch.Tensor], target_names: list[str], pad_idx: int = 0, average: str = 'macro') -> dict:
    """Вычисляет метрики precision, recall, f1-score для каждой целевой переменной"""

    predictions, targets = normalize_sizes(predictions, targets, target_names)
    metrics_dict = {}
    
    for key in target_names:
        # Получаем предсказанные индексы классов
        _, pred_indices = predictions[key].max(dim=-1)
        
        pred_np = pred_indices.to('cpu').numpy()
        target_np = targets[key].to('cpu').numpy()
        
        # Создаем маску для игнорирования pad_idx
        mask = target_np != pad_idx
        
        # Фильтруем паддинг
        pred_filtered = pred_np[mask]
        target_filtered = target_np[mask]
        
        # Вычисляем метрики
        precision, recall, f1, support = precision_recall_fscore_support(target_filtered, pred_filtered, average=average, zero_division=0)
        
        # Вычисляем accuracy (для полноты)
        accuracy = (pred_filtered == target_filtered).mean()
        
        metrics_dict[key] = {
            'accuracy': accuracy,
            'precision': float(precision),
            'recall': float(recall),
            'f1': float(f1)}
    
    return metrics_dict

In [11]:
train_df = pd.read_parquet(os.path.join(DATASET_PATH, 'prepared_ru_syntagrus-ud-train.parquet'))
validation_df = pd.read_parquet(os.path.join(DATASET_PATH, 'prepared_ru_syntagrus-ud-dev.parquet'))
test_df = pd.read_parquet(os.path.join(DATASET_PATH, 'prepared_ru_syntagrus-ud-test.parquet'))

In [12]:
train_df.head(2)

,source_words,tokens,input_ids,upos_ids,head_ids,deprel_ids,Mood_ids,NumType_ids,VerbForm_ids,ExtPos_ids,...,Variant_ids,Number_ids,Gender_ids,NumForm_ids,Aspect_ids,Case_ids,PronType_ids,Tense_ids,Abbr_ids,Voice_ids
0,"[Анкета, .]","[[А, ##нк, ##ета], [.]]","[[3, 4, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[3, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[3, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[3, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",...,"[3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[3, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[3, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[3, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,"[Начальник, областного, управления, связи, Сем...","[[Нача, ##ль, ##ник], [обла, ##ст, ##ного], [у...","[[7, 8, 9, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[3, 5, 3, 3, 6, 6, 7, 3, 5, 4, 8, 9, 3, 10, 10...","[5, 6, 4, 6, 4, 7, 5, 3, 5, 8, 5, 9, 8, 8, 8, ...","[5, 6, 7, 7, 8, 9, 10, 3, 6, 4, 11, 12, 13, 14...","[3, 3, 3, 3, 3, 3, 4, 3, 3, 3, 4, 3, 3, 3, 3, ...","[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, ...","[3, 3, 3, 3, 3, 3, 4, 3, 3, 3, 4, 3, 3, 3, 3, ...","[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, ...",...,"[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, ...","[3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3, 4, 3, 4, 4, ...","[5, 6, 6, 3, 5, 5, 5, 5, 5, 4, 5, 4, 3, 4, 4, ...","[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, ...","[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3, 3, 3, 3, ...","[3, 5, 5, 5, 3, 3, 4, 3, 3, 4, 4, 4, 6, 4, 4, ...","[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3, ...","[3, 3, 3, 3, 3, 3, 4, 3, 3, 3, 4, 3, 3, 3, 3, ...","[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, ...","[3, 3, 3, 3, 3, 3, 4, 3, 3, 3, 4, 3, 3, 3, 3, ..."


In [13]:
with open('data/vocabs_configuration.json', 'r', encoding='utf-8') as file:
    vocabs_config = json.load(file)

MAX_WORDS_COUNT = vocabs_config['MAX_WORDS_COUNT']
MAX_SUBTOKENS_COUNT = vocabs_config['MAX_SUBTOKENS_COUNT']
MAX_LETTERS_COUNT = vocabs_config['MAX_LETTERS_COUNT']
SOURCE_VOCAB_LEN = vocabs_config['SOURCE_VOCAB_LEN']
LETTERS_VOCAB_LEN = vocabs_config['LETTERS_VOCAB_LEN']
TRG_VOCABS_LEN = vocabs_config['TRG_VOCABS_LEN']
PAD_IDX = vocabs_config['PAD_IDX']

In [14]:
target_names = ['upos', 'head', 'deprel', 'Mood', 'NumType', 'VerbForm',
       'ExtPos', 'Reflex', 'Polarity', 'Typo', 'NameType', 'InflClass',
       'Person', 'Poss', 'Animacy', 'Degree', 'Foreign', 'Variant', 'Number',
       'Gender', 'NumForm', 'Aspect', 'Case', 'PronType', 'Tense', 'Abbr', 'Voice']
source_name = 'source_text'

In [15]:
# target_weights = {key : {} for key in target_names}

# dataframes = [train_df, test_df, validation_df]

# empty_nodes_counter = 0

# for target_name in target_names:
#     target_weights[target_name]['grammem_weight'] = 1.0 # Вес, вложимый конкретной граммемой в суммарную ошибку
#     total_labels_count = 0
#     labels_count = {}
#     for df_idx, dataframe in enumerate(dataframes):
#         for i in range(len(dataframe)):
#             row = dataframe.loc[i, target_name]
#             for label in row:
#                 if label == '_':
#                     # print(f'dataframe {df_idx}\n index {i}')
#                     empty_nodes_counter += 1
#                 total_labels_count += 1
#                 if label not in labels_count:
#                     labels_count[label] = 1
#                 else:
#                     labels_count[label] += 1
    
#     target_classes_weights = [1]*TRG_VOCABS_LEN[target_name]
#     if USE_CLASSES_WEIGHTS:
#         for key in labels_count.keys():
#             labels_count[key] = labels_count[key] / total_labels_count
#             target_classes_weights[target_vocabs[target_name].token_to_idx[key]] = CLASSES_WEIGHTS_SCALER / (math.exp(CLASSES_WEIGHTS_SCALER * labels_count[key]))
#     target_weights[target_name]['classes_weights'] = torch.tensor(target_classes_weights, dtype=torch.float32)

#     print('='*40)
#     print(target_name)
#     print(f'target_weights {target_weights[target_name]['classes_weights']}')
#     print(f'classes_distribution {labels_count}')

In [16]:
if USE_PRETRAINED:
    with open(f"data/{WORD_REPRESENTATION}_train_states.json", "r", encoding="utf-8") as file:
        train_states = json.load(file)
        training_epochs = int(train_states[-1]['training_epochs'])

    with open(f"data/{WORD_REPRESENTATION}_validation_states.json", "r", encoding="utf-8") as file:
        validation_states = json.load(file)
    
    model = torch.load(MODEL_SAVE_FILEPATH, weights_only=False)
else:
    train_states = []
    validation_states = []
    training_epochs = 0
    model = MHAModel(MAX_WORDS_COUNT, MAX_SUBTOKENS_COUNT, MAX_LETTERS_COUNT, LETTERS_VOCAB_LEN, SOURCE_VOCAB_LEN, TOKENS_EMBEDDING_DIM, LETTERS_EMBEDDING_DIM,\
                     MAIN_ATTENTION_DIM, MAIN_NUM_HEADS, MAIN_NUM_ENCODER_LAYERS, CLASSIFIER_FC_HIDDEN_DIM, MAIN_ENCODER_FC_HIDDEN_DIM,\
                     TRG_VOCABS_LEN, WORDS_POS_ENCODING, WORD_SUBTOKENS_POS_ENCODING, LETTERS_POS_ENCODING, ROPE_BASE,\
                     LETTERS_IN_WORD_ATTENTION_DIM, DROPOUT, TEMPERATURE, BATCH_FIRST, WORD_REPRESENTATION, INIT_WEIGHTS, BIAS, PAD_IDX, DEVICE)

In [17]:
dataset = CustomDataset(train_df, target_names, MAX_SUBTOKENS_COUNT, MAX_WORDS_COUNT,\
                        MAX_LETTERS_COUNT, valid_df=validation_df)

In [18]:
model = model.to(device=DEVICE)
optimizer = optim.AdamW(model.parameters(), LEARNING_RATE, weight_decay=WEIGHT_DECAY)
# sheduler = OneCycleLR(optimizer, MAX_LEARNING_RATE, epochs=EPOCHS, steps_per_epoch=BATCH_COUNT)

In [19]:
try:
    for epoch in range(EPOCHS):
        train_start_time = time.time()
        training_epochs += 1
        print('='*50)
        print(f'Epoch {training_epochs}')
        dataset.set_dataframe_split('train')
        batch_generator = generate_batches(dataset, BATCH_SIZE, SHUFFLE, DROP_LAST, DEVICE)
        epoch_sum_train_loss = 0.0
        epoch_running_train_loss = 0.0
        train_epoch_metrics = {key:{'accuracy' : 0.0, 'precision' : 0.0, 'recall' : 0.0, 'f1' : 0.0, 'mean_loss' : 0.0} for key in target_names}
        model.train()
        for batch_idx, batch_dict in enumerate(batch_generator):

            optimizer.zero_grad()
            
            if WORD_REPRESENTATION == 'tokens':
                predictions = model(tokens=batch_dict['input_ids'], letters=None)
            elif WORD_REPRESENTATION == 'letters':
                predictions = model(tokens=None, letters=batch_dict['letters'])
            else:
                predictions = model(tokens=batch_dict['input_ids'], letters=batch_dict['letters'])

            # total_loss, train_losses = compute_loss(predictions, batch_dict, target_names, target_weights, PAD_IDX)
            total_loss, train_losses = compute_loss(predictions, batch_dict, target_names, PAD_IDX)

            total_loss.backward()

            optimizer.step()
            # sheduler.step()

            # Метрики
            epoch_running_train_loss += (total_loss.item() - epoch_running_train_loss) / (batch_idx + 1)
            epoch_sum_train_loss += total_loss.item()

            cur_metrics = compute_metrics(predictions, batch_dict, target_names, PAD_IDX)
            for key in target_names:
                for metric, value in cur_metrics[key].items():
                    train_epoch_metrics[key][metric] += (value - train_epoch_metrics[key][metric]) / (batch_idx + 1)
                train_epoch_metrics[key]['mean_loss'] += (train_losses[key].item() - train_epoch_metrics[key]['mean_loss']) / (batch_idx + 1)
        train_end_time = time.time()

        dataset.set_dataframe_split('validation')
        batch_generator = generate_batches(dataset, BATCH_SIZE, SHUFFLE, DROP_LAST, DEVICE)
        epoch_sum_valid_loss = 0.0
        epoch_running_valid_loss = 0.0
        valid_epoch_metrics = {key:{'accuracy' : 0.0, 'precision' : 0.0, 'recall' : 0.0, 'f1' : 0.0, 'mean_loss' : 0.0} for key in target_names}
        model.eval()
        valid_start_time = time.time()

        with torch.no_grad():
            for batch_idx, batch_dict in enumerate(batch_generator):
                
                if WORD_REPRESENTATION == 'tokens':
                    predictions = model(tokens=batch_dict['input_ids'], letters=None)
                elif WORD_REPRESENTATION == 'letters':
                    predictions = model(tokens=None, letters=batch_dict['letters'])
                else:
                    predictions = model(tokens=batch_dict['input_ids'], letters=batch_dict['letters'])

                # total_loss, valid_losses = compute_loss(predictions, batch_dict, target_names, target_weights, PAD_IDX)
                total_loss, valid_losses = compute_loss(predictions, batch_dict, target_names, PAD_IDX)

                # Средние потери и точность
                epoch_running_valid_loss += (total_loss.item() - epoch_running_valid_loss) / (batch_idx + 1)
                epoch_sum_valid_loss += total_loss.item()

                cur_metrics = compute_metrics(predictions, batch_dict, target_names, PAD_IDX)
                for key in target_names:
                    for metric, value in cur_metrics[key].items():
                        valid_epoch_metrics[key][metric] += (value - valid_epoch_metrics[key][metric]) / (batch_idx + 1)
                    valid_epoch_metrics[key]['mean_loss'] += (valid_losses[key].item() - valid_epoch_metrics[key]['mean_loss']) / (batch_idx + 1)
        valid_end_time = time.time()

        train_states.append(train_epoch_metrics)
        train_states[-1]['summed loss'] = epoch_sum_train_loss
        train_states[-1]['training_epochs'] = training_epochs
        train_states[-1]['execution_time'] = train_end_time - train_start_time

        validation_states.append(valid_epoch_metrics)
        validation_states[-1]['summed loss'] = epoch_sum_valid_loss
        validation_states[-1]['training_epochs'] = training_epochs
        validation_states[-1]['execution_time'] = valid_end_time - valid_start_time
        
        print(f'Train: Средняя ошибка эпохи {epoch_running_train_loss}')
        for key in target_names:
            print('-'*20)
            print(f'Train: Ошибка на признаке {key}: {train_epoch_metrics[key]['mean_loss']}')
            print(f'Train: Точность на признаке {key}: {train_epoch_metrics[key]['accuracy']*100}%')
            print(f'Train: precision на признаке {key}: {train_epoch_metrics[key]['precision']*100}%')
            print(f'Train: recall на признаке {key}: {train_epoch_metrics[key]['recall']*100}%')
            print(f'Train: f1-score на признаке {key}: {train_epoch_metrics[key]['f1']*100}%')
        print(f'Время выполнения {train_end_time - train_start_time}')

        print('-'*40)
        print(f'Validation: Средняя ошибка эпохи {epoch_running_valid_loss}')
        for key in target_names:
            print('-'*20)
            print(f'Validation: Ошибка на признаке {key}: {valid_epoch_metrics[key]['mean_loss']}')
            print(f'Validation: Точность на признаке {key}: {valid_epoch_metrics[key]['accuracy']*100}%')
            print(f'Validation: precision на признаке {key}: {valid_epoch_metrics[key]['precision']*100}%')
            print(f'Validation: recall на признаке {key}: {valid_epoch_metrics[key]['recall']*100}%')
            print(f'Validation: f1-score на признаке {key}: {valid_epoch_metrics[key]['f1']*100}%')
        print(f'Время выполнения {valid_end_time - valid_start_time}')

except KeyboardInterrupt:
    print('Принудительная остановка')

Epoch 31
Train: Средняя ошибка эпохи 0.8089611702641509
--------------------
Train: Ошибка на признаке upos: 0.03106340781046322
Train: Точность на признаке upos: 98.83731146410571%
Train: precision на признаке upos: 97.16602586752882%
Train: recall на признаке upos: 96.94169322824389%
Train: f1-score на признаке upos: 96.96983761515241%
--------------------
Train: Ошибка на признаке head: 0.3597076058277984
Train: Точность на признаке head: 89.26698620403312%
Train: precision на признаке head: 78.8879648856144%
Train: recall на признаке head: 78.95443398517352%
Train: f1-score на признаке head: 77.91661630087899%
--------------------
Train: Ошибка на признаке deprel: 0.18945376970162045
Train: Точность на признаке deprel: 93.70843941318152%
Train: precision на признаке deprel: 84.39851482164948%
Train: recall на признаке deprel: 82.65950615928675%
Train: f1-score на признаке deprel: 82.88766105800055%
--------------------
Train: Ошибка на признаке Mood: 0.002263345401004108
Train: Точ

In [20]:
dfdf

NameError: name 'dfdf' is not defined

In [21]:
save_results_to_file(model, MODEL_SAVE_FILEPATH, train_states, validation_states)

In [ ]:
dataframes = [train_df, test_df, validation_df]

empty_nodes_counter = 0

for target_name in target_names:
    total_labels_count = 0
    labels_count = {}
    for df_idx, dataframe in enumerate(dataframes):
        for i in range(len(dataframe)):
            row = dataframe[target_name].iloc[i]
            for label in row:
                # if label == '_':
                #     print(f'dataframe {df_idx}\n index {i}')
                #     empty_nodes_counter += 1
                total_labels_count += 1
                if label not in labels_count:
                    labels_count[label] = 1
                else:
                    labels_count[label] += 1
    
    for key in labels_count.keys():
        labels_count[key] = labels_count[key] / total_labels_count * 100

    print(f'classes_distribution {labels_count}')

    # print(empty_nodes_counter)

classes_distribution {'NOUN': 23.876469824737793, 'PUNCT': 18.491115001288065, 'ADJ': 9.263050119812911, 'PROPN': 3.3061379657039245, 'AUX': 0.905267749078755, 'VERB': 11.388386877439, 'ADP': 9.288020751319523, 'ADV': 5.03392118636195, 'CCONJ': 3.6295043494359867, 'PART': 3.3167455426763905, 'PRON': 4.359911792397201, 'DET': 3.6069055984946465, 'SCONJ': 1.8796494624066813, 'NUM': 1.1799446956514863, '_': 0.14514591348038552, 'INTJ': 0.015219566960494353, 'X': 0.2302700715451418, 'SYM': 0.08433353120966568}


In [ ]:
# Исследование ошибок модели
try:
    mistake_count = 0
    with torch.no_grad():
        for idx in range(len(validation_df)):
            row = validation_df.iloc[idx]
            vectorized_dict = vectorizer.vectorize(row, target_names, MAX_SUBTOKENS_COUNT, MAX_WORDS_COUNT, MAX_LETTERS_COUNT, ADD_BOS_EOS_TOKENS)

            # Тензоры для входа и подсчёта субтокенов
            source_x = torch.tensor(vectorized_dict['src_vectorized']).unsqueeze(0).to(DEVICE)
            subtokens_cnt = torch.tensor(vectorized_dict['subtokens_cnt']).unsqueeze(0).to(DEVICE)
            letters = torch.tensor(vectorized_dict['letters']).unsqueeze(0).to(DEVICE)

            # Тензоры целевых меток
            trg_tensors = {key: torch.tensor(vectorized_dict['trg_vectorized'][key]).unsqueeze(0).to(DEVICE)
                        for key in target_names}

            predictions = model(source_x, subtokens_cnt, letters)

            # Индексы предсказанных меток
            pred_indices = {}
            for key in target_names:
                _, idx_max = predictions[key].max(dim=-1)
                pred_indices[key] = idx_max.squeeze(0)

            # Ошибки
            mistakes = reveal_mistakes(predictions, trg_tensors, target_names, pad_idx)

            for key in target_names:
                mistake_tensor = mistakes[key].squeeze(0).float()
                if mistake_tensor.sum() > 0:
                    mistake_count += 1
                    print('-'*30)
                    print(f'{mistake_count}. Исходное предложение: {row["source_text"]}. Индекс в датафрейме = {idx}. Количество ошибок: {mistake_tensor.sum()}')
                    for i in range(mistake_tensor.size(0)):
                        if mistake_tensor[i] > 0:
                            word = row['source_words'][i]
                            windowed_word = ' '.join(row['source_words'][i-1:i+2])
                            predicted_label = target_vocabs[key].idx_to_token[pred_indices[key][i].item()]
                            print(f'Ошибка в " {windowed_word} ".  Слово " {word} ", граммема {key}')
                            print(f'Токенизированное слово: {tokenizer.encode(word).tokens}')
                            print(f'Предсказанная метка: {predicted_label}, реальная метка: {row[key][i]}')
                            print(f'Пропорция предсказанной метки {labels_count[predicted_label]}, Пропорция реальной метки {labels_count[row[key][i]]}')
except KeyboardInterrupt:
    print('Принудительная остановка')

------------------------------
1. Исходное предложение: Алгоритм, от имени учёного аль-Хорезми, - точный набор инструкций, описывающих порядок действий исполнителя для достижения результата решения задачи за конечное время.. Индекс в датафрейме = 0. Количество ошибок: 2.0
Ошибка в " имени учёного аль ".  Слово " учёного ", граммема upos
Токенизированное слово: ['уч', '##ё', '##ного']
Предсказанная метка: ADJ, реальная метка: NOUN
Пропорция предсказанной метки 9.263050119812911, Пропорция реальной метки 23.876469824737793
Ошибка в " учёного аль - ".  Слово " аль ", граммема upos
Токенизированное слово: ['аль']
Предсказанная метка: ADV, реальная метка: PART
Пропорция предсказанной метки 5.03392118636195, Пропорция реальной метки 3.3167455426763905
------------------------------
2. Исходное предложение: Это связано с тем, что работа каких-то инструкций алгоритма может быть зависима от других инструкций или результатов их работы.. Индекс в датафрейме = 2. Количество ошибок: 1.0
Ошибка в " 